In [ ]:
# =============================================================
# PHASTPADDING / ARFP — FAITHFUL (Chang et al. 2025, IEEE TNSRE) on LEMON Mu/Alpha
#   Ported from PHASTPADDING_FAITHFUL_theta_eval to the LEMON mu/alpha setup so it sits
#   alongside EPN / PEAP / BOB: same LEMON cache, same FIR-231 9-13 Hz GT, same eval
#   times, same metrics, same JSON layout (join on 'name'). Algorithm is UNCHANGED.
# =============================================================
# Forecast-padding pipeline (Chang et al. 2025, Sec. II.A / II.E.1) -- identical to the theta nb:
#   1. signal = bandpass(raw window);  noise = raw window - signal
#   2. AR-forecast the SIGNAL forward (edge-trim -> aryule -> forward-predict)
#   3. padding = signal_forecast + LOCAL noise baseline (DC);  append to the RAW window
#   4. bandpass again (FIR-231) -> Hilbert -> phase at t=0
#
# The LEMON port changes ONLY the framework, never the algorithm:
#   • data       : LEMON .npz cache (single-channel C3 Hjorth, 2D epochs, native 1000 Hz,
#                  resting -> PRE_SAMP=0); chronological cal/test split (first cal_n = cal).
#   • band / GT  : 4-8 Hz FIR-175  ->  9-13 Hz FIR-231 (byte-identical to the EPN/PEAP GT).
#   • eval times : [0.3..0.7] (onset-locked)  ->  [1.5..3.5] (from epoch start).
#   • SNR        : precomputed ch_theta_snr  ->  alpha_snr_per_epoch (9-13/1-40 power), like EPN/PEAP.
#   • params     : values that were sample-scaled to 300 Hz are expressed in ms and re-sampled to FS
#                  (EDGE 65 ms; INPUT_WIN 980 ms; N_FORECAST=(taps-1)//2=115; AR order 30; BASE_MS 120 ms).
#   • output     : result_alpha/PHASTPADDING_lemon/phastpadding_alpha_results.json,
#                  + n_train_samples (=0, no training) and compute_time_ms, to match EPN/PEAP keys.
#   theta_* descriptors are renamed alpha_*; fail_pct is kept (PHASTPADDING-specific robustness stat).

import numpy as np
import os, time, json, gc, glob, warnings
from scipy.signal import firwin, filtfilt, hilbert, welch
from scipy.stats import pearsonr, spearmanr
warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print(f"(Skipping Drive mount: {e})")

# ---- DATA + OUTPUT (LEMON, matched to EPN/PEAP) ----
LOAD_DIR = '/content/drive/MyDrive/RAIN_IRC/LEMON'
OUTDIR   = '/content/drive/MyDrive/RAIN_IRC/result_alpha/PHASTPADDING_lemon'   # parallel to EPN_lemon / PEAP_lemon
os.makedirs(OUTDIR, exist_ok=True)

files = sorted(glob.glob(os.path.join(LOAD_DIR, '*.npz')))
assert files, f"no npz found in {LOAD_DIR}"
_d0 = np.load(files[0], allow_pickle=True)
FS = float(_d0['fs']); PRE_SAMP = 0; _d0.close()      # resting epochs: t=0 at epoch start

EVAL_TIMES_S    = [1.5, 2.0, 2.5, 3.0, 3.5]
EVAL_TIMES_SAMP = [PRE_SAMP + int(round(t * FS)) for t in EVAL_TIMES_S]
ALPHA_SEARCH_LO, ALPHA_SEARCH_HI = 7.0, 14.0

# --- GT == forecast-padding's acausal filter: 9-13 Hz zero-phase FIR-231 (filtfilt) ---
# firwin(231,[9,13],hamming,fs=1000) -> byte-identical taps to the EPN/PEAP LEMON GT.
GT_BAND, GT_FIR_NUMTAPS = [9.0, 13.0], 231
gt_fir_b = firwin(GT_FIR_NUMTAPS, GT_BAND, fs=FS, window='hamming', pass_zero=False)

# --- forecast-padding params (paper; ms-based values re-sampled to FS) ---
INPUT_WIN  = int(round(0.980 * FS))             # 980 ms input length                 -> 980 @ 1000 Hz
EDGE_MS    = 65                                  # phastimate edge: 65 ms @ 1 kHz (~20 samp @ 300 Hz)
EDGE       = int(round(EDGE_MS / 1000.0 * FS))   #                                     -> 65 @ 1000 Hz
AR_ORDER   = 30                                  # phastimate standard AR order
N_FORECAST = (GT_FIR_NUMTAPS - 1) // 2           # = 115 samp (~115 ms), covers the FIR-231 acausal reach
BASE_MS    = 120                                 # local-baseline window for the noise padding (~one alpha+ cycle)

print(f"\n{'='*94}")
print(f"PHASTPADDING / ARFP — FAITHFUL (Chang et al. 2025) on LEMON Mu/Alpha — {FS:.0f} Hz, FIR-231 GT")
print(f"{'='*94}")
print(f"  FS={FS:.0f} Hz | t=0 at sample {PRE_SAMP} (resting)")
print(f"  forecast = phastimate-style AR (order {AR_ORDER}, edge {EDGE} samp = {EDGE_MS} ms) on the FILTERED signal "
      f"-> {N_FORECAST} samp (~{N_FORECAST/FS*1000:.0f} ms; covers the FIR-231 reach)")
print(f"  pipeline: signal forecast + LOCAL noise baseline (BASE_MS={BASE_MS} ms) -> append to RAW {INPUT_WIN}-samp "
      f"window -> filtfilt(FIR-231 9-13) -> Hilbert @ t=0")
print(f"  Data: {LOAD_DIR}  ({len(files)} subjects) | Output: {OUTDIR}")


# =============================================================
# forecast-padding internals                              [ALGORITHM UNCHANGED]
# =============================================================

def aryule(x, order):
    """MATLAB aryule: Levinson-Durbin on biased autocorrelation. a=[1,a1,...,ap]."""
    x = np.asarray(x, np.float64); n = len(x)
    r = np.correlate(x, x, 'full')[n - 1:n + order] / n
    a = np.zeros(order + 1); a[0] = 1.0; e = r[0]
    if e <= 1e-12:
        return a
    for m in range(1, order + 1):
        acc = r[m] + (np.dot(a[1:m], r[1:m][::-1]) if m > 1 else 0.0)
        k = -acc / e
        ao = a[1:m].copy(); a[1:m] = ao + k * ao[::-1]; a[m] = k
        e *= (1.0 - k * k)
        if e <= 1e-12:
            break
    return a

def _ff(b, x):
    return filtfilt(b, [1.0], x, padlen=min(3 * len(b), len(x) - 1))

def phastpad_predict(raw_1d, t0, input_window, fir_b, edge, ord, n_forecast, base_ms=None):
    """FAITHFUL forecast padding / ARFP (Chang et al. 2025). Forecast the FILTERED signal,
    shift it to the raw window's LOCAL baseline (noise padding), append to the RAW window,
    bandpass again, Hilbert, read phase at t=0. The baseline shift removes the DC/drift-driven
    discontinuity at the t=0 read point."""
    if base_ms is None:
        base_ms = BASE_MS
    start = t0 - input_window + 1
    if start < 0:
        return np.nan
    window = raw_1d[start:t0 + 1].astype(np.float64)        # RAW window, keep its baseline
    try:
        sig = _ff(fir_b, window)                            # signal = bandpassed rhythm
    except Exception:
        return np.nan
    sige = sig[edge:len(sig) - edge]
    if len(sige) < ord + 5 or np.std(sige) < 1e-10:
        return np.nan
    a = aryule(sige, ord); coeffs = -np.flip(a[1:]); L = len(sige)
    ext = np.empty(L + edge + n_forecast); ext[:L] = sige
    for k in range(edge + n_forecast):                      # forecast the SIGNAL (filtered domain)
        p = L + k; ext[p] = np.dot(coeffs, ext[p - ord:p])
    if not np.all(np.isfinite(ext)):
        return np.nan
    cap = 6.0 * np.std(sige)
    fc = np.clip(ext[L + edge:L + edge + n_forecast], -cap, cap)   # rhythm forecast for t>0
    nb = max(1, int(round(base_ms / 1000.0 * FS)))
    noise = window - sig
    baseline = float(np.mean(noise[-nb:]))                  # local pre-t0 baseline -> absorbs DC & drift
    padding = fc + baseline                                 # signal forecast shifted to raw baseline
    padded = np.concatenate([window, padding])              # smooth join (no zero-mean discontinuity)
    try:
        pf = _ff(fir_b, padded)                             # second convolution (bandpass removes DC itself)
    except Exception:
        return np.nan
    idx = input_window - 1                                  # phase at t=0
    if idx >= len(pf):
        return np.nan
    return float(np.angle(hilbert(pf)[idx]))


# =============================================================
# GT + metrics + alpha descriptors  (EPN/PEAP/BOB-IDENTICAL)
# =============================================================

def gt_phase_fir(epoch):
    try:
        return np.angle(hilbert(_ff(gt_fir_b, epoch.astype(np.float64))))
    except Exception:
        return np.full(len(epoch), np.nan)

def circ(a, b):
    return np.angle(np.exp(1j * (a - b)))

def errors_to_metrics(errors):
    errors = np.asarray(errors, float); errors = errors[np.isfinite(errors)]
    if len(errors) < 10:
        return {'mace': 90.0, 'plv': 0.0, 'csd': 180.0, 'acc': 0.0, 'bias': 0.0, 'n_samples': 0}
    ma = float(np.mean(np.abs(errors))); R = float(np.abs(np.mean(np.exp(1j * errors))))
    return {'mace': float(np.degrees(ma)), 'plv': R,
            'csd': float(np.degrees(np.sqrt(-2.0 * np.log(max(R, 1e-12))))),
            'acc': float((1.0 - ma / np.pi) * 100.0),
            'bias': float(np.degrees(np.angle(np.mean(np.exp(1j * errors))))),
            'n_samples': int(len(errors))}

def itpc_at(ph_list, trials, esamp):
    vals = [ph_list[tr][esamp] for tr in trials
            if esamp < len(ph_list[tr]) and np.isfinite(ph_list[tr][esamp])]
    return float(np.abs(np.mean(np.exp(1j * np.array(vals))))) if len(vals) >= 2 else float('nan')

def alpha_snr_per_epoch(raw_1ch):
    """mean over epochs of (9-13 Hz power / 1-40 Hz power); identical to EPN/PEAP / the LEMON loader's mu-SNR."""
    snrs = []
    for tr in range(raw_1ch.shape[0]):
        ep = raw_1ch[tr].astype(np.float64)
        psd = np.abs(np.fft.rfft(ep)) ** 2
        freqs = np.fft.rfftfreq(len(ep), 1.0 / FS)
        mu = psd[(freqs >= 9) & (freqs <= 13)].mean()
        broad = psd[(freqs >= 1) & (freqs <= 40)].mean()
        snrs.append(mu / broad if broad > 0 else 0.0)
    return float(np.mean(snrs))

def estimate_alpha_peak(raw_1ch, onset_samp):
    seg = raw_1ch[:, onset_samp:]
    if seg.shape[1] < 16:
        return np.nan
    nfft = int(2 ** np.ceil(np.log2(seg.shape[1] * 4))); psd = None
    for tr in range(seg.shape[0]):
        f, px = welch(seg[tr].astype(np.float64), fs=FS, nperseg=seg.shape[1], nfft=nfft, detrend='linear')
        psd = px if psd is None else psd + px
    psd /= seg.shape[0]; band = (f >= ALPHA_SEARCH_LO) & (f <= ALPHA_SEARCH_HI)
    return float(f[band][np.argmax(psd[band])]) if np.any(band) else np.nan

def run_per_tp(raw, ph_fir, trials, epoch_len):
    per_tp = {t: [] for t in EVAL_TIMES_S}; n_eval = n_fail = 0
    for tr in trials:
        for t, es in zip(EVAL_TIMES_S, EVAL_TIMES_SAMP):
            if es >= epoch_len or es - INPUT_WIN + 1 < 0:
                continue
            n_eval += 1
            est = phastpad_predict(raw[tr], es, INPUT_WIN, gt_fir_b, EDGE, AR_ORDER, N_FORECAST)
            gf = ph_fir[tr][es]
            if not (np.isfinite(est) and np.isfinite(gf)):
                n_fail += 1; continue
            per_tp[t].append(circ(est, gf))
    return {t: np.array(v) for t, v in per_tp.items()}, (100.0 * n_fail / max(n_eval, 1))


Mounted at /content/drive

PHASTPADDING / ARFP — FAITHFUL (Chang et al. 2025) on LEMON Mu/Alpha — 1000 Hz, FIR-231 GT
  FS=1000 Hz | t=0 at sample 0 (resting)
  forecast = phastimate-style AR (order 30, edge 65 samp = 65 ms) on the FILTERED signal -> 115 samp (~115 ms; covers the FIR-231 reach)
  pipeline: signal forecast + LOCAL noise baseline (BASE_MS=120 ms) -> append to RAW 980-samp window -> filtfilt(FIR-231 9-13) -> Hilbert @ t=0
  Data: /content/drive/MyDrive/RAIN_IRC/LEMON  (20 subjects) | Output: /content/drive/MyDrive/RAIN_IRC/result_alpha/PHASTPADDING_lemon


In [ ]:
# =============================================================
# Main loop  [forecast-padding fixed pipeline; LEMON chronological TEST split, per timepoint]
# =============================================================
print(f"\n{'Subj':<14s}{'N':>5s}{'MACE':>7s}{'PLV':>7s}{'CSD':>7s}{'Acc':>7s}{'bias':>7s}{'pk':>5s}{'SNR':>6s}{'ITPC':>6s}{'fail%':>6s}{'t':>5s}")
print("-" * 96)

results = []
for fp in files:
    t0 = time.time(); d = np.load(fp, allow_pickle=True); name = os.path.basename(fp)[:-4]
    if float(d['fs']) != FS:
        print(f"{name}: fs mismatch, skipping"); d.close(); continue
    raw = d['epochs'].astype(np.float64)               # LEMON: single channel, 2D (n_trials x n_samp)
    n_trials, epoch_len = raw.shape
    ch_name = str(d['montage']) if 'montage' in d.files else 'C3-Hjorth'
    cal_n = int(d['cal_n'])
    test_trials = np.arange(cal_n, n_trials)           # chronological split (first cal_n = cal)
    alpha_snr = alpha_snr_per_epoch(raw)

    ph_fir = [gt_phase_fir(raw[tr]) for tr in range(n_trials)]
    itpc_tp = {t: (itpc_at(ph_fir, test_trials, es) if es < epoch_len else float('nan'))
               for t, es in zip(EVAL_TIMES_S, EVAL_TIMES_SAMP)}
    itpc_mean = float(np.nanmean(list(itpc_tp.values())))
    f_peak = estimate_alpha_peak(raw, PRE_SAMP)

    tp, fail = run_per_tp(raw, ph_fir, test_trials, epoch_len)
    pool = (np.concatenate([tp[t] for t in EVAL_TIMES_S])
            if any(len(tp[t]) for t in EVAL_TIMES_S) else np.array([]))
    m = errors_to_metrics(pool); tp_m = {t: errors_to_metrics(tp[t]) for t in EVAL_TIMES_S}

    # per-call latency (analogue of EPN/PEAP compute_time_ms; 50 reps of one prediction)
    tsamps = []
    valid_es = [es for es in EVAL_TIMES_SAMP if es < epoch_len and es - INPUT_WIN + 1 >= 0]
    if len(test_trials) and valid_es:
        tr0, es0 = int(test_trials[0]), valid_es[0]
        for _ in range(50):
            ts = time.perf_counter()
            phastpad_predict(raw[tr0], es0, INPUT_WIN, gt_fir_b, EDGE, AR_ORDER, N_FORECAST)
            tsamps.append((time.perf_counter() - ts) * 1000)

    dt = time.time() - t0; pk = f"{f_peak:.1f}" if np.isfinite(f_peak) else "n/a"
    print(f"{name:<14s}{m['n_samples']:>5d}{m['mace']:>6.1f}\u00b0{m['plv']:>7.3f}{m['csd']:>6.1f}\u00b0"
          f"{m['acc']:>6.1f}%{m['bias']:>+6.1f}\u00b0{pk:>5s}{alpha_snr:>6.2f}{itpc_mean:>6.3f}{fail:>5.1f}%{dt:>4.0f}s")

    results.append({
        'name': name, 'best_ch': ch_name, 'n_trials': int(n_trials),
        'n_test_samples': m['n_samples'], 'n_train_samples': 0, 'fail_pct': fail,
        'alpha_snr': alpha_snr,
        'alpha_itpc_per_timepoint': {str(t): itpc_tp[t] for t in EVAL_TIMES_S},
        'alpha_itpc_mean': itpc_mean,
        'alpha_peak_hz': float(f_peak) if np.isfinite(f_peak) else None,
        'metrics_pooled': m,
        'metrics_per_timepoint': {str(t): tp_m[t] for t in EVAL_TIMES_S},
        'errors_per_timepoint_rad': {str(t): [float(x) for x in tp[t]] for t in EVAL_TIMES_S},
        'compute_time_ms': float(np.mean(tsamps)) if tsamps else 0.0})
    d.close(); del raw, ph_fir; gc.collect()

print(f"\n{len(results)} subjects done")



Subj              N   MACE    PLV    CSD    Acc   bias   pk   SNR  ITPC fail%    t
------------------------------------------------------------------------------------------------
sub-032311      405  30.1°  0.773  41.1°  83.3%  +1.2° 11.0  3.60 0.093  0.0%   3s
sub-032317      410  34.3°  0.730  45.4°  80.9%  +4.1° 10.1  3.60 0.083  0.0%   5s
sub-032324      400  27.8°  0.787  39.7°  84.5%  +0.8° 10.6  3.90 0.087  0.0%   5s
sub-032330      430  36.8°  0.710  47.4°  79.6%  +8.7° 11.1  3.76 0.134  0.0%   5s
sub-032332      400  30.8°  0.785  39.9°  82.9% +10.3° 10.4  3.49 0.077  0.0%   5s
sub-032355      430  33.2°  0.749  43.6°  81.6%  +5.6° 10.4  3.52 0.077  0.0%   5s
sub-032361      405  26.8°  0.821  35.9°  85.1%  +5.9° 10.5  4.32 0.135  0.0%   5s
sub-032370      400  26.1°  0.826  35.5°  85.5%  -3.4° 10.9  3.58 0.122  0.0%   5s
sub-032383      420  29.5°  0.790  39.4°  83.6%  +5.2° 10.3  4.87 0.055  0.0%   5s
sub-032394      420  35.8°  0.698  48.6°  80.1%  -0.5° 11.3  3.96 0.081 

In [ ]:
# =============================================================
# SUMMARY + SAVE  (model-tagged JSON, EPN/PEAP-parallel: single fixed condition)
# =============================================================
valid = [r for r in results if r['n_test_samples'] >= 10]; n = len(valid)
M = np.array([r['metrics_pooled']['mace'] for r in valid])
P = np.array([r['metrics_pooled']['plv'] for r in valid])
C = np.array([r['metrics_pooled']['csd'] for r in valid])
A = np.array([r['metrics_pooled']['acc'] for r in valid])
B = np.array([r['metrics_pooled']['bias'] for r in valid])
SNR = np.array([r['alpha_snr'] for r in valid]); IT = np.array([r['alpha_itpc_mean'] for r in valid])

print('=' * 72); print('PHASTPADDING (FAITHFUL) ALPHA (mu) — fixed pipeline (single condition)'); print('=' * 72)
print(f"  MACE {M.mean():.1f}\u00b0\u00b1{M.std():.1f}   PLV {P.mean():.3f}   CSD {C.mean():.1f}\u00b0   Acc {A.mean():.1f}%")
print(f"  bias {B.mean():+.1f}\u00b0 (mean) | per-subject |bias| mean {np.mean(np.abs(B)):.1f}\u00b0")
print(f"  mean failure rate {np.mean([r['fail_pct'] for r in results]):.1f}%")
print(f"  (n={n} participants; RAW errors, no bias correction)")

print('\n--- Per-timepoint ---')
print(f"{'ms':>5}{'MACE':>8}{'PLV':>8}{'CSD':>8}{'Acc':>8}{'bias':>8}{'ITPC':>8}{'N':>7}")
for t in EVAL_TIMES_S:
    g = lambda k: np.array([r['metrics_per_timepoint'][str(t)][k] for r in valid])
    ti = np.array([r['alpha_itpc_per_timepoint'][str(t)] for r in valid])
    print(f"{int(t*1000):>5}{g('mace').mean():>7.1f}\u00b0{g('plv').mean():>8.3f}{g('csd').mean():>7.1f}\u00b0"
          f"{g('acc').mean():>7.1f}%{g('bias').mean():>+7.1f}\u00b0{np.nanmean(ti):>8.3f}{g('n_samples').mean():>7.0f}")

group_stats = {'n_participants': int(n),
               'mean_mace': float(M.mean()), 'median_mace': float(np.median(M)),
               'mean_plv': float(P.mean()), 'mean_csd': float(C.mean()), 'mean_acc': float(A.mean()),
               'mean_bias': float(B.mean()), 'mean_abs_bias': float(np.mean(np.abs(B))),
               'mean_fail_pct': float(np.mean([r['fail_pct'] for r in results]))}

print('\n--- Accuracy correlations across subjects ---')
if n >= 3 and np.std(SNR) > 0:
    r1, p1 = pearsonr(SNR, A); rs1, ps1 = spearmanr(SNR, A)
    group_stats.update({'acc_vs_snr_pearson_r': float(r1), 'acc_vs_snr_pearson_p': float(p1),
                        'acc_vs_snr_spearman_r': float(rs1), 'acc_vs_snr_spearman_p': float(ps1)})
    print(f"  Acc vs SNR : r={r1:+.3f} (p={p1:.4f})  rho={rs1:+.3f} (p={ps1:.4f})")
if n >= 3 and np.std(IT) > 0:
    r2, p2 = pearsonr(IT, A); rs2, ps2 = spearmanr(IT, A)
    group_stats.update({'acc_vs_itpc_pearson_r': float(r2), 'acc_vs_itpc_pearson_p': float(p2),
                        'acc_vs_itpc_spearman_r': float(rs2), 'acc_vs_itpc_spearman_p': float(ps2)})
    print(f"  Acc vs ITPC: r={r2:+.3f} (p={p2:.4f})  rho={rs2:+.3f} (p={ps2:.4f})")

out = {'model': 'PHASTPADDING',
       'config': {'fs': FS, 'pre_samp': PRE_SAMP, 'eval_times_s': EVAL_TIMES_S, 'horizon_ms': 0.0,
                  'gt_band': GT_BAND, 'gt_fir_numtaps': GT_FIR_NUMTAPS,
                  'input_window': INPUT_WIN, 'edge': EDGE, 'edge_ms': EDGE_MS, 'ar_order': AR_ORDER,
                  'n_forecast': N_FORECAST, 'base_ms': BASE_MS,
                  'method': ('Chang et al. 2025 (IEEE TNSRE) forecast padding / ARFP: bandpass -> AR-forecast the '
                             'signal -> shift forecast to local raw baseline (noise padding) -> append to raw window '
                             '-> bandpass again (FIR-231) -> Hilbert @ t=0'),
                  'forecast_engine': 'filter -> edge-trim -> aryule (Yule-Walker) -> forward-predict the filtered signal',
                  'noise_padding': 'predicted signal shifted to local raw baseline (DC of raw-signal over last BASE_MS ms)',
                  'acausal_filter': 'FIR-231 9-13 Hz Hamming filtfilt (== GT filter)',
                  'data_dir': LOAD_DIR, 'dataset': 'LEMON top-20 mu-SNR (C3 Hjorth)', 'band': 'mu/alpha 9-13 Hz',
                  'split': 'chronological cal/test (first cal_n epochs = cal)',
                  'channel': 'C3 Hjorth (single channel)', 'conditions': ['single (fixed pipeline)'],
                  'itpc_basis': 'GT-phase ITPC, test trials, per eval point', 'errors_unit': 'radians',
                  'no_training': True,
                  'note': ('faithful forecast-padding (Chang et al. 2025) ported from the theta/onset notebook to '
                           'LEMON mu/alpha; fixed paper params, no per-subject training -> n_train_samples=0. '
                           'Headline metrics under metrics_pooled / metrics_per_timepoint / errors_per_timepoint_rad '
                           '(analogue of BOB opt_*). alpha_snr / alpha_itpc_* / alpha_peak_hz are computed identically '
                           'to EPN/PEAP on the native epochs + shared FIR-231 GT, so they match per subject across '
                           'models. fail_pct = % of eval points with no finite estimate (PHASTPADDING-specific). '
                           'Same data/GT/split as EPN/PEAP/BOB LEMON -> join on name.'),
                  'bias_note': 'RAW errors (no correction); residual systematic bias is the paper bandpass shift'},
       'group_stats': group_stats, 'subjects': results}
outpath = os.path.join(OUTDIR, 'phastpadding_alpha_results.json')
with open(outpath, 'w') as f:
    json.dump(out, f, indent=2, default=str)
print(f"\nSaved -> {outpath}")
print("Per subject: metrics_pooled (incl. bias) + per-timepoint + errors (rad) + SNR/ITPC/peak + fail_pct + compute_time_ms.")
print("Same data/GT/split as EPN/PEAP/BOB LEMON -> join on 'name';"
      " PHASTPADDING.metrics_pooled <-> EPN.metrics_pooled <-> BOB.opt_metrics_pooled.")


PHASTPADDING (FAITHFUL) ALPHA (mu) — fixed pipeline (single condition)
  MACE 30.2°±4.2   PLV 0.785   CSD 39.7°   Acc 83.2%
  bias +4.6° (mean) | per-subject |bias| mean 6.3°
  mean failure rate 0.0%
  (n=20 participants; RAW errors, no bias correction)

--- Per-timepoint ---
   ms    MACE     PLV     CSD     Acc    bias    ITPC      N
 1500   31.1°   0.776   40.5°   82.7%   +4.5°   0.081     84
 2000   31.6°   0.772   41.0°   82.4%   +5.8°   0.113     84
 2500   29.3°   0.798   38.2°   83.7%   +5.0°   0.086     84
 3000   29.2°   0.791   38.9°   83.8%   +2.6°   0.102     84
 3500   29.6°   0.793   38.7°   83.5%   +4.9°   0.104     84

--- Accuracy correlations across subjects ---
  Acc vs SNR : r=+0.615 (p=0.0039)  rho=+0.266 (p=0.2567)
  Acc vs ITPC: r=-0.126 (p=0.5967)  rho=-0.071 (p=0.7672)

Saved -> /content/drive/MyDrive/RAIN_IRC/result_alpha/PHASTPADDING_lemon/phastpadding_alpha_results.json
Per subject: metrics_pooled (incl. bias) + per-timepoint + errors (rad) + SNR/ITPC/peak 